# BNPP fraud detection

*Date: 20/04/2026*

*Author: Dylan LEBRETON*

## Imports

In [ ]:
import polars as pl
import polars.selectors as cs

# No columns truncation
_ = pl.Config.set_tbl_cols(-1)

from IPython.display import display

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

import pandas as pd

from sklearn.preprocessing import OrdinalEncoder

from typing import Optional, Union, List, Literal

from itertools import combinations

# plotly renderer
pio.renderers.default = "notebook"

In [ ]:
# Useful function I developed at work
def numeric_to_intervals(
        expr: pl.Expr,
        bounds: List[Union[float, int]],
        inclusive: Literal['right', 'left'] = "right",
        fmt: Optional[str] = None,
) -> pl.Expr:
    
    import numpy as np
    
    def format_number(value: Union[int, float], fmt: str = "") -> str:
        if not fmt:
            return str(value)

        if fmt.endswith('h'):
            num_spec = fmt[:-1] or ".2f"
            abs_value = abs(value)

            suffixes = [
                (1e12, 'T'),
                (1e9, 'B'),
                (1e6, 'M'),
                (1e3, 'K'),
            ]

            for factor, suffix in suffixes:
                if abs_value >= factor:
                    scaled = abs_value / factor
                    formatted = format(scaled, num_spec)
                    sign = "-" if value < 0 else ""
                    return f"{sign}{formatted}{suffix}"

            return format(value, num_spec)

        return format(value, fmt)
    
    if inclusive not in ["right", "left"]:
        raise ValueError(
            f"Unknown `inclusive` parameter. Accepted: 'left', 'right'. Received: {inclusive}."
        )

    bounds = [bound for bound in sorted(list(set(bounds))) if bound not in [-np.inf, np.inf]]
    if len(bounds) == 0:
        raise ValueError("`bounds` is empty or only contains useless values (as np.inf).")

    ordered_intervals = []

    # ----------------------------------
    # First interval (-inf to first bound)
    # ----------------------------------

    # If right bound should be included in the interval
    if inclusive == "right":
        result = pl.when(expr <= bounds[0])
        right_bracket = "]"
    else:
        result = pl.when(expr < bounds[0])
        right_bracket = "["

    right_bound = format_number(bounds[0], fmt=fmt) if fmt else bounds[0]
    first_interval = f"]-∞, {right_bound}{right_bracket}"
    result = result.then(pl.lit(first_interval))
    ordered_intervals.append(first_interval)

    # --------------------------------------------------------
    # If only two intervals have been defined (one bound given)
    # --------------------------------------------------------
    if len(bounds) == 1:
        left_bound = format_number(bounds[0], fmt=fmt) if fmt else bounds[0]
        left_bracket = "]" if inclusive == "right" else "["
        last_interval = f"{left_bracket}{left_bound}, +∞["
        result = result.otherwise(pl.lit(last_interval))
        ordered_intervals.append(last_interval)

    # --------------------------------------------------------
    # Other case (more than two intervals)
    # --------------------------------------------------------
    else:
        # Middle intervals
        for i, bound in enumerate(bounds[1:], start=1):
            if inclusive == "right":
                result = result.when((bounds[i - 1] < expr) & (expr <= bounds[i]))
                left_bracket = "]"
                right_bracket = "]"
            else:
                result = result.when((bounds[i - 1] <= expr) & (expr < bounds[i]))
                left_bracket = "["
                right_bracket = "["

            left_bound = (
                format_number(bounds[i - 1], fmt=fmt)
                if fmt
                else bounds[i - 1]
            )
            right_bound = (
                format_number(bounds[i], fmt=fmt) if fmt else bounds[i]
            )
            interval = f"{left_bracket}{left_bound}, {right_bound}{right_bracket}"
            result = result.then(pl.lit(interval))
            ordered_intervals.append(interval)

        # Last interval
        left_bound = (
            format_number(bounds[-1], fmt=fmt) if fmt else bounds[-1]
        )
        left_bracket = "]" if inclusive == "right" else "["
        last_interval = f"{left_bracket}{left_bound}, +∞["
        result = result.otherwise(pl.lit(last_interval))
        ordered_intervals.append(last_interval)

    try:
        result_with_nans = pl.when(expr.is_null() | expr.is_nan()).then(pl.lit(None)).otherwise(result)
    except Exception:  # noqa
        result_with_nans = pl.when(expr.is_null()).then(pl.lit(None)).otherwise(result)

    return result_with_nans.cast(pl.Enum(ordered_intervals))

## Data Preparation

We load the data.

In [ ]:
x_train = pl.read_csv(r"./data/X_train_G3tdtEn.csv", infer_schema_length=None)
y_train = pl.read_csv(r"./data/Y_train_2_XPXJDyy.csv", infer_schema_length=None)

To facilitate the study of the x_train data, we must unpivot the items columns into single ones (adding an item rank column).

In [ ]:
# Check no duplicate IDs
print(f"Rows : {x_train.height}")
print(f"Unique IDs : {x_train['ID'].n_unique()}")
print(f"Duplicate IDs : {x_train.height - x_train['ID'].n_unique()}")
assert x_train.height == x_train["ID"].n_unique()

# Definition of a function to make the multiple unpivot
def multi_unpivot(df: pl.DataFrame) -> pl.DataFrame:
    """
    Function to unpivot all the items columns in the data
    """
    result = []
    for item_rank in range(1, 24+1):
        item_df = (
            df
            .select(
                pl.col(['ID', 'Nb_of_items']),
                pl.lit(item_rank).alias('item_rank'),
                *[pl.col(f'{prefix}{item_rank}').alias(prefix) for prefix in ["item", "cash_price", "make", "model", "goods_code", "Nbr_of_prod_purchas"]]
            )
        )
        result.append(item_df)
    return pl.concat(result, how='vertical_relaxed')

In [ ]:
# Application on x_train
x_train = (
    x_train
    .pipe(multi_unpivot)
    .sort('ID', 'item_rank')
)

The unpivot created rows for each empty item column, we delete these rows.

In [ ]:
# Definition of a function to remove the empty items rows
def remove_empty_items_rows(df: pl.DataFrame) -> pl.DataFrame:
    """
    Function to remove empty items rows after unpivot.
    """
    return (
        df
        .filter(
            pl.any_horizontal(pl.col(["item", "cash_price", "make", "model", "goods_code", "Nbr_of_prod_purchas"]).is_not_null())
        )
    )

In [ ]:
# Application on x_train
x_train = (
    x_train
    .pipe(remove_empty_items_rows)
)

We add the target to x_train (to prevent joining it all the time in data analysis sections).

In [ ]:
train = (
    x_train
    .join(
        y_train.drop("index"),
        on="ID",
        how="left"
    )
    .with_columns(
        pl.col('fraud_flag').alias('target'),
    )
    .drop('fraud_flag')
)

We rename some columns names to snake case.

In [ ]:
train = (
    train
    .rename(
        dict(
            Nb_of_items="n_items",
            Nbr_of_prod_purchas="n_prods"
        )
    )
)

And we correct some data types.

In [ ]:
train = (
    train
    .with_columns(
        pl.col('ID').cast(pl.Utf8),
        pl.col('n_items').cast(pl.Int64),
        pl.col('n_prods').cast(pl.Int64)
    )
)

## Exploratory Data Analysis

### Summary

We print the summary of the data (and equivalent for categorical columns).

In [ ]:
display(train.describe())
display(train.select(cs.by_dtype(pl.Utf8)).to_pandas().describe(include='all'))

<div style="background:AliceBlue">

- `ID`: count (92,790) matches the challenge specification.
- `n_items`: some baskets are truncated (max = 60) $\implies$ how many? Error? Flag to identify them?
- `item_rank`: most baskets contain only a few items (median = 1).
- `item`: 173 unique categories, dominated by `COMPUTERS` (~31% of items). High cardinality $\implies$ grouping/encoding.
- `cash_price`: highly skewed (mean €701, std €742, max €21,995) $\implies$ outliers and €0 values to investigate. Unit price to add (with `n_prods`)?
- `make`: 1,273 missing values (~0.8%) and 829 unique categories. Dominated by `APPLE` (~48% of non-null rows). High cardinality $\implies$ grouping/encoding.
- `model`: 1,273 missing values (~0.8%, same as missing `make`?) and 9,679 unique categories. Dominated by `RETAILER` (~27% of non-null rows). High cardinality $\implies$ grouping/encoding.
- `goods_code`: 14,384 unique codes, dominated by `FULFILMENT` (~15%, default value for unprocessed items?). No missing values $\implies$ could help fill `make` and `model`?
- `n_prods`: highly skewed (median = 1, max = 40). High counts could be a fraud signal?

</div>

### Number of items per basket

Analysis of number of items by basket.

In [ ]:
df = (
    train
    .group_by('ID')
    .agg(
        pl.col('n_items').first(),
        pl.col('target').first()
    )
)

In [ ]:
px.histogram(df, x="n_items", color="target", nbins=100, marginal="box", title="Number of item per basket vs target").show()

In [ ]:
px.histogram(df, x="n_items", color="target", histnorm="probability", nbins=100, marginal="box", title="Share of item per basket vs target").show()

<div style="background:AliceBlue">

- The majority of baskets contain few items (2.9 on average).
- 87% of fraud baskets are small baskets (1 to 3 items).
- Truncated baskets (containing more theoretical items than the expected 24) are all non-fraudulent $\implies$ no need of a flag.

</div>

In [ ]:
# Baskets where n_items is high
train.sort('n_items', descending=True).head()

<div style="background:AliceBlue">

Baskets with many items appear to represent home decoration. Nothing to clean.

</div>

In [ ]:
# Fraud rate per basket size
(
    df
    .group_by('n_items')
    .agg([
        pl.len().alias('n_baskets'),
        pl.col('target').sum().alias('n_frauds'),
        pl.col('target').mean().alias('fraud_rate') * 100,
    ])
    .sort('n_items')
    .with_columns(pl.col('n_items').cast(pl.String))
    .transpose(include_header=True, header_name='n_items per basket', column_names='n_items')
)

<div style="background:AliceBlue">

- The highest fraud rate is 1.87% (higher than the overall 1.4%) on baskets with 2 items.
- For baskets with 19 items, the rate climbs to 17%. However, baskets with more than 6 items have few items (<200) and very little fraud (<= 2). The high variance then explains the significant fraud rates.

</div>

In [ ]:
# with pl.Config(tbl_rows=-1):
#     display(
#         train
#         .filter(
#             (pl.col('n_items') == 19) & (pl.col('target') == 1)
#         )
#     )

<div style="background:AliceBlue">

Large fraudulent orders do not seem abnormal.

</div>

### Number of products

#### Per item

In [ ]:
# Distribution of n_prods
px.histogram(train, x='n_prods', nbins=100, marginal="box", title="Number of product per item").show()

In [ ]:
px.histogram(train, x='n_prods', histnorm='probability', nbins=100, title="Share of product per item").show()

<div style="background:AliceBlue">

97% of items have only one product. 

</div>

We can compare with the distribution of products for items involved in fraud baskets.

In [ ]:
# Fraud rate per number of products
(
    train
    .group_by('n_prods')
    .agg([
        pl.len().alias('n_items'),
        pl.col('target').sum().alias('n_frauds'),
        pl.col('target').mean().alias('fraud_rate') * 100,
    ])
    .sort('n_prods')
    .with_columns(pl.col('n_prods').cast(pl.String))
    .transpose(include_header=True, header_name='n_prods per item', column_names='n_prods')
)

In [ ]:
print(f"Share of fraud items that have one product: {train.filter((pl.col('target') == 1) & (pl.col('n_prods') == 1)).height / train.filter(pl.col('target') == 1).height * 100:.2f}") 

<div style="background:AliceBlue">

- The fraud rate (per item !) is 1.43% for items with a single product. Rates for items with more products are unreliable due to the small number of items linked to fraud (only 38 items for items with 2 products).
- 98% of fraudulent items have one product, but 97% of items have one product originally.

The fraud therefore exclusively concerns baskets containing 1 to 3 items, each item generally containing only one product.

However, since the majority of items contain only one product, it is necessary to examine the number of products per basket.

</div>

Before examining the number of products per basket, let's see the baskets having items with a lot of products to verify the presence of outliers in non-fraud baskets.

In [ ]:
(
    train
    .filter(
        (pl.col('n_prods') >= 30).any().over('ID')
    )
)

<div style="background:AliceBlue">

Baskets containing items with many products appear legitimate.

</div>

#### Per basket

We compute a dataframe at basket-level.

In [ ]:
df = (
    train
    .group_by('ID')
    .agg(
        pl.col('n_prods').sum(),
        pl.col('target').first(),
        pl.col('n_items').first()
    )
)

df

And we can compute now the distributions.

In [ ]:
px.histogram(df, x='n_prods', nbins=100, marginal='box', title='Number of product per basket').show()
px.histogram(df, x='n_prods', nbins=100, histnorm='probability', title='Share of product per basket').show()

<div style="background:AliceBlue">

As expected, since most baskets contain one to three items and each item typically contains one product, we generally find 1 to 3 products per basket:
- 51% have one product
- 33% have two products
- 9% have three products
- 7% have four or more products

</div>

Let's compare the distributions to number of frauds.

In [ ]:
# Share of frauds per number of products
(
    df
    .filter(pl.col('target') == 1)
    ['n_prods']
    .value_counts()
    .sort('n_prods')
    .with_columns(
        pl.col('count').sum().alias('total')
    )
    .with_columns(
        (pl.col('count') / pl.col('total') * 100.0).alias('share')
    )
    .with_columns(
        pl.col('n_prods').cast(pl.Utf8)
    )
    .transpose(include_header=True, header_name='n_prods per basket', column_names='n_prods')
)

In [ ]:
# Fraud rates per products
(
    df
    .group_by('n_prods')
    .agg([
        pl.len().alias('n_items'),
        pl.col('target').sum().alias('n_frauds'),
        pl.col('target').mean().alias('fraud_rate') * 100,
    ])
    .sort('n_prods')
    .with_columns(pl.col('n_prods').cast(pl.String))
    .transpose(include_header=True, header_name='n_prods per basket', column_names='n_prods')
)

<div style="background:AliceBlue">

- 46% of fraudulent baskets contain only one product, and 45% contain two products.
- But fraud rate is higher for baskets with two products (1.88%) than one product (1.28%).

A basket with two items of one product is a strong signal in favor of fraud.

</div>

### Item categories

#### Cleaning

We can look at every values available in `item` column.

In [ ]:
# with pl.Config(tbl_rows=-1):
#     display(
#         train['item'].value_counts().sort('count', descending=True)
#     )

The column needs to be cleaned. We propose a preliminary cleaning before analysis.

In [ ]:
train = (
    train
        
    # Deletion of characters
    .with_columns(
        pl.col('item')

        # Everyone to uppercase
        .str.to_uppercase()

        # Deletion of any character not being A-Z or number
        .str.replace_all(r'[^A-Z0-9]', '')

        .alias('item_clean')
    )

    # Merge of bathroom items
    .with_columns(
        pl.col('item_clean')
        .replace_strict(
            {
                "BATHROOMACCESSORIES": "BATHROOM",
                "BATHROOMFIXTURES": "BATHROOM"
            },
            default=pl.col('item_clean')
        )
        .alias('item_clean')
    )

    # Drop items starting with digits
    .filter(
        ~pl.col('item_clean').str.contains(r'^\d+[A-Z]')
    )

    # Set some outliers to "OTHER"
    .with_columns(
        pl.when(
            pl.col('item_clean').is_in(
                [
                    'APPLEPRODUCTDESCRIPTION',
                    'APPLES',
                    'HPELITEBOOK850V6',
                    'LOGITECHPEBBLEM350BLUETOOTHWIRELESSMOUSE',
                    'MICROSOFTOFFICEHOMEANDSTUDENT2019',
                    'PRODUCT',
                    'TARGUSGEOLITEESSENTIALCASE',
                    'TOSHIBAPORTABLEHARDDRIVE',
                    'UNKNOWN'
                ]
            )
        )
        .then(pl.lit('OTHER'))
        .otherwise(pl.col('item_clean'))
        .alias('item_clean')
    )
)

# # Display of new values
# with pl.Config(tbl_rows=-1):
#     display(train['item_clean'].value_counts().sort('item_clean'))

<div style="background:AliceBlue">

**To go further**:

In production, and if the `item` column proves to be a strong predictor of the target, we could:
- define more general categories (based on frequencies for example)
- assign each `item` value to one of these categories using third-party algorithms (fuzzy matching, small LLM)

</div>

We replace each low-frequency value with "OTHER".

In [ ]:
train = (
    train
    .with_columns(
        pl.when(pl.col('item_clean').count().over('item_clean') < 20)
        .then(pl.lit('OTHER'))
        .otherwise(pl.col('item_clean'))
        .alias('item_clean')
    )
)

#### Study

We compute several metrics based on items combinations over baskets.

In [ ]:
items_combinations = (
    train
    .group_by('ID', 'target')
    .agg(
        pl.col('item_clean').unique(),
        pl.col('n_items').first(),
        pl.col('n_prods').sum(),
    )
    .with_columns(
        pl.col('item_clean').list.sort()
    )
    .group_by('item_clean', pl.col('target').cast(pl.Utf8).replace_strict({'1': 'fraud_n_baskets', '0': 'no_fraud_n_baskets'}), pl.col('n_items'), pl.col('n_prods'))
    .agg(
        pl.len().alias('n_baskets')
    )
    .pivot(
        on='target',
        index=['item_clean', 'n_items', 'n_prods'],
        values='n_baskets'
    )
    .fill_null(0)
    .with_columns(
        (pl.col('fraud_n_baskets') + pl.col('no_fraud_n_baskets')).alias('n_baskets')
    )
    .with_columns(
        pl.col('n_baskets').sum().alias('total_n_baskets')
    )
    .with_columns(
        fraud_rate = pl.col('fraud_n_baskets') / pl.col('n_baskets') * 100.0,
        basket_share = pl.col('n_baskets') / pl.col('total_n_baskets') * 100.0
    )
    .with_columns(
        total_fraud_n_baskets = pl.col('fraud_n_baskets').sum(),
    )
    .with_columns(
        fraud_basket_share = pl.col('fraud_n_baskets') / pl.col('total_fraud_n_baskets') * 100.0
    )
)

In [ ]:
display(items_combinations.sort('n_baskets', descending=True).head(10))
display(items_combinations.sort('fraud_n_baskets', descending=True).head(10))

<div style="background:AliceBlue">

- The majority of baskets are for electronic purchases.
- The purchase of a computer alone accounts for 27% of baskets and 41% of fraudulent baskets.

The purchase of a computer alone with fulfilment charges accounts for 11% of baskets and 25% of fraudulent baskets. The fraud rate increases from 2.14% to 3% when fulfilment charges are added to the purchase of a computer.

</div>

In [ ]:
# Fraud rate comparison vs FULFILMENTCHARGE
(
    train
    .group_by('ID', 'target')
    .agg(
        (pl.col('item_clean') == "FULFILMENTCHARGE").any().alias('has_charge'),
    )
    .group_by('has_charge', pl.col('target').cast(pl.Utf8).replace_strict({'1': 'fraud_n_baskets', '0': 'no_fraud_n_baskets'}))
    .agg(
        pl.len().alias('n_baskets')
    )
    .pivot(
        on='target',
        values='n_baskets',
        index='has_charge'
    )
    .with_columns(
        n_baskets = pl.col('no_fraud_n_baskets') + pl.col('fraud_n_baskets')
    )
    .with_columns(
        fraud_rate = pl.col('fraud_n_baskets') / pl.col('n_baskets') * 100.0
    )
)

In [ ]:
# Fraud rate comparison vs COMPUTERS
(
    train
    .group_by('ID', 'target')
    .agg(
        (pl.col('item_clean') == "COMPUTERS").any().alias('has_computer'),
    )
    .group_by('has_computer', pl.col('target').cast(pl.Utf8).replace_strict({'1': 'fraud_n_baskets', '0': 'no_fraud_n_baskets'}))
    .agg(
        pl.len().alias('n_baskets')
    )
    .pivot(
        on='target',
        values='n_baskets',
        index='has_computer'
    )
    .with_columns(
        n_baskets = pl.col('no_fraud_n_baskets') + pl.col('fraud_n_baskets')
    )
    .with_columns(
        fraud_rate = pl.col('fraud_n_baskets') / pl.col('n_baskets') * 100.0
    )
)

In [ ]:
# Fraud rate comparison vs COMPUTERS & FULFILMENT CHARGES
(
    train
    .group_by('ID', 'target')
    .agg(
        pl.col('item_clean').unique(),
    )
    .with_columns(
        has_both = pl.col('item_clean').list.contains('COMPUTERS') & pl.col('item_clean').list.contains('FULFILMENTCHARGE')
    )
    
    .group_by('has_both', pl.col('target').cast(pl.Utf8).replace_strict({'1': 'fraud_n_baskets', '0': 'no_fraud_n_baskets'}))
    .agg(
        pl.len().alias('n_baskets')
    )
    .pivot(
        on='target',
        values='n_baskets',
        index='has_both'
    )
    .with_columns(
        n_baskets = pl.col('no_fraud_n_baskets') + pl.col('fraud_n_baskets')
    )
    .with_columns(
        fraud_rate = pl.col('fraud_n_baskets') / pl.col('n_baskets') * 100.0
    )
)

<div style="background:AliceBlue">

- Fraud rate increases from 0.53% to 2.19% when the basket contains a computer
- 1.13% to 2.22% when the basket contains fulfiment charge
- 1.17 to 2.75 when the basket contains both

The conjoint presence of a computer and fulfiment charge is a strong indictor in favor of fraud.
</div>

### Make

We display the values available in `make`.

In [ ]:
# with pl.Config(tbl_rows=-1):
#     display(train['make'].value_counts().sort('count', descending=True))

<div style="background:AliceBlue">

- We see 1,273 null values, to treat.
- As for `item`, we see a lot of wrong values caused by characters.
- The most frequent values are electronics make as "APPLE", "SAMSUNG", "LG". We also see "RETAILER" which should be linked to the "FULFILMENTCHARGE" items.
   
</div>

#### Cleaning

First, let's see the baskets concerned by null values for `make`.

In [ ]:
# with pl.Config(tbl_rows=-1):
#     display(
#         train
#         .filter(
#             (pl.col('make').is_null().any()).over('ID')
#         )
#     )

<div style="background:AliceBlue">

`make` and `model` are null on the same rows. The corresponding items look normal.
   
</div>

We check if we can interpolate the columns based on `goods_code` (which has no null values). First, we check if `goods_code` has no duplicate make or model.

In [ ]:
# (
#     train
#     .filter(
#         pl.col('make').is_not_null()
#     )
#     .group_by('goods_code')
#     .agg(
#         pl.col('make').unique(),
#         pl.col('model').unique()
#     )
#     .filter(
#         (pl.col('make').list.len() > 1) | (pl.col('model').list.len() > 1)
#     )
# )

<div style="background:AliceBlue">

Similar `goods_code` can be associated with different `make` or `model` values, and "RETAILER" is generally present as default value. We propose to replace make and model by their mode in goods_code except RETAILER.
   
</div>

In [ ]:
make_filling_base = (
    train
    .filter(pl.col('make').is_not_null()) 
    .group_by('goods_code', 'make')
    .agg(pl.len())
    .with_columns(
        is_non_retailer=(pl.col('make') != 'RETAILER').cast(pl.Int8)
    )
    .sort(['goods_code', 'is_non_retailer', 'len'], descending=[False, True, True])
    .unique('goods_code', keep='first')
    .drop('is_non_retailer')
)

In [ ]:
train = (
    train
    .join(
        make_filling_base
        .select('goods_code', 'make'),
        on='goods_code',
        how='left'
    )
    .with_columns(
        pl.when(
            pl.col('make').is_not_null()
        )
        .then(
            pl.col('make')
        )
        .otherwise(
            pl.col('make_right')
        )
        .alias('make_clean')
    )
    .drop('make_right')
)

# with pl.Config(tbl_rows=-1):
#     display(
#         train
#         .filter(pl.col('make_clean').is_null())
#     )

<div style="background:AliceBlue">

There are only 109 remaining rows where `make` and `model` are nulls. No fraud on the corresponding baskets, we decide to delete those baskets.
   
</div>


In [ ]:
train = (
    train
    .filter(
        pl.col('make_clean').is_not_null().all().over('ID')
    )
)

Now, we clean the values of the `make` column as we did for `item`.

In [ ]:
train = (
    train
    .with_columns(
        pl.col('make_clean')

        # Everyone to uppercase
        .str.to_uppercase()

        # Deletion of any character not being A-Z or number
        .str.replace_all(r'[^A-Z0-9]', '')

        .alias('make_clean')
    )
    .with_columns(
        pl.when(pl.col('make_clean').count().over('make_clean') < 20)
        .then(pl.lit('OTHER'))
        .otherwise(pl.col('make_clean'))
        .alias('make_clean')
    )
)

#### Study

In [ ]:
makes_combinations = (
    train
    .group_by('ID', 'target')
    .agg(
        pl.col('make_clean').unique(),
        pl.col('n_items').first(),
        pl.col('n_prods').sum(),
    )
    .with_columns(
        pl.col('make_clean').list.sort()
    )
    .group_by('make_clean', pl.col('target').cast(pl.Utf8).replace_strict({'1': 'fraud_n_baskets', '0': 'no_fraud_n_baskets'}), pl.col('n_items'), pl.col('n_prods'))
    .agg(
        pl.len().alias('n_baskets')
    )
    .pivot(
        on='target',
        index=['make_clean', 'n_items', 'n_prods'],
        values='n_baskets'
    )
    .fill_null(0)
    .with_columns(
        (pl.col('fraud_n_baskets') + pl.col('no_fraud_n_baskets')).alias('n_baskets')
    )
    .with_columns(
        pl.col('n_baskets').sum().alias('total_n_baskets')
    )
    .with_columns(
        fraud_rate = pl.col('fraud_n_baskets') / pl.col('n_baskets') * 100.0,
        basket_share = pl.col('n_baskets') / pl.col('total_n_baskets') * 100.0
    )
    .with_columns(
        total_fraud_n_baskets = pl.col('fraud_n_baskets').sum(),
    )
    .with_columns(
        fraud_basket_share = pl.col('fraud_n_baskets') / pl.col('total_fraud_n_baskets') * 100.0
    )
)

In [ ]:
display(makes_combinations.sort('n_baskets', descending=True).head(10))
display(makes_combinations.sort('fraud_n_baskets', descending=True).head(10))

<div style="background:AliceBlue">

- 56% of the baskets are for the purchase of an Apple product. 36% are for a standalone purchase, and 20% are with a "RETAILER" order which seems linked to the FULFILMENT CHARGE mentioned earlier.
- Fraud primarily affects these two types of baskets, as they represent 43% and 32% of fraudulent baskets.

However, a higher fraud rate is observed on baskets containing two Apple products:
- 1.68% on a single Apple product
- 2.28% on Apple + Retailer products
- 2.51% on two Apple products

Higher fraud rates are also observed on other products such as two-Samsung (5.96%), but the basket volumes involved are small (9 baskets).

</div>

Let's check the link between "RETAILER" and "FULFILMENTCHARGE".

In [ ]:
df = (
    train
    .filter(pl.col('target') == 1)
    .filter(pl.col('make_clean') == 'RETAILER')
    .group_by('item_clean')
    .agg(pl.col('ID').n_unique().alias('n_baskets'))
    .sort('n_baskets', descending=False)
)
px.bar(df.tail(10), x='n_baskets', y='item_clean', text='n_baskets', title='Number of baskets for each item being in basket with "RETAILER" make').show()

In [ ]:
(
    train
    .filter(
        pl.col('item_clean') == 'FULFILMENTCHARGE'
    )
    ['make_clean']
    .value_counts()
)

<div style="background:AliceBlue">

"RETAILER" items in fraud baskets are almost exclusively "FULFILMENTCHARGE" (552) and "WARRANTY" (21).

</div>

### Model

#### Cleaning

We display the values available in `model`.

In [ ]:
# train['model'].value_counts().sort('count', descending=True)

<div style="background:AliceBlue">

- The `model` variable has a very high cardinality (9,679 distinct values).
- The values are very heterogeneous, even for similar products (sizes, quotation marks, punctuation, truncated versions, etc.).

We decide to clean the column using goods codes.

</div>

In [ ]:
model_filling_base = (
    train
    .filter(pl.col('model').is_not_null()) 
    .group_by('goods_code', 'model')
    .agg(pl.len())
    .with_columns(
        is_non_retailer=(pl.col('model') != 'RETAILER').cast(pl.Int8)
    )
    .sort(['goods_code', 'is_non_retailer', 'len'], descending=[False, True, True])
    .unique('goods_code', keep='first')
    .drop('is_non_retailer')
)

In [ ]:
train = (
    train
    .join(
        model_filling_base
        .select('goods_code', 'model'),
        on='goods_code',
        how='left'
    )
    .with_columns(
        pl.when(
            pl.col('model').is_not_null()
        )
        .then(
            pl.col('model')
        )
        .otherwise(
            pl.col('model_right')
        )
        .alias('model_clean')
    )
    .drop('model_right')
)

# with pl.Config(tbl_rows=-1):
#     display(
#         train
#         .filter(pl.col('model_clean').is_null())
#     )

#### Study

In [ ]:
models_combinations = (
    train
    .group_by('ID', 'target')
    .agg(
        pl.col('model_clean').unique(),
        pl.col('n_items').first(),
        pl.col('n_prods').sum(),
    )
    .with_columns(
        pl.col('model_clean').list.sort()
    )
    .group_by('model_clean', pl.col('target').cast(pl.Utf8).replace_strict({'1': 'fraud_n_baskets', '0': 'no_fraud_n_baskets'}), pl.col('n_items'), pl.col('n_prods'))
    .agg(
        pl.len().alias('n_baskets')
    )
    .pivot(
        on='target',
        index=['model_clean', 'n_items', 'n_prods'],
        values='n_baskets'
    )
    .fill_null(0)
    .with_columns(
        (pl.col('fraud_n_baskets') + pl.col('no_fraud_n_baskets')).alias('n_baskets')
    )
    .with_columns(
        pl.col('n_baskets').sum().alias('total_n_baskets')
    )
    .with_columns(
        fraud_rate = pl.col('fraud_n_baskets') / pl.col('n_baskets') * 100.0,
        basket_share = pl.col('n_baskets') / pl.col('total_n_baskets') * 100.0
    )
    .with_columns(
        total_fraud_n_baskets = pl.col('fraud_n_baskets').sum(),
    )
    .with_columns(
        fraud_basket_share = pl.col('fraud_n_baskets') / pl.col('total_fraud_n_baskets') * 100.0
    )
)

In [ ]:
display(models_combinations.sort('n_baskets', descending=True).head(10))
display(models_combinations.sort('fraud_n_baskets', descending=True).head(10))

<div style="background:AliceBlue">

- 38% of fraudulent baskets are for macbooks.
- Fraud rate for macbook pro 14 is 6%.
- Fraud rate for ipad pro is 20%, but the number of fraudulent baskets is low (29 vs 120 for macbook).

</div>

### Goods codes

We display the values available in `goods_code`.

In [ ]:
(
    train
    .group_by('goods_code')
    .agg(
        pl.col('ID').n_unique().alias('n_baskets'),
        pl.col('model_clean').unique()
    )
    .sort('n_baskets', descending=True)
)

<div style="background:AliceBlue">

- First goods_code is FULFILMENT (15% of good_codes).
- The following codes are naturally all the computer codes (especially macbooks) studied previously.

</div>

### Item prices

*Looking at the products, `cash_price` is the total price of an item (including the number of products). The corresponding code has been intentionally deleted from the notebook.*

`cash_price` showed a highly skewed distribution, with some €0 values and extreme outliers. We investigate these specific cases, then analyze prices at the basket level and their relationship with fraud.

#### Zero prices

We first check the share of items with a €0 price, then investigate what kind of items they correspond to.

In [ ]:
# display(
#     train
#     .filter(pl.col('cash_price') < 1.0)
#     ['item_clean']
#     .value_counts()
# )

<div style="background:AliceBlue">

The item "FULFILMENTCHARGE" was examined in the section dedicated to the variable `item`. The item "SERVICE" was also frequent. It most likely corresponds to an item similar to "FULFILMENTCHARGE" or "WARRANTY".

</div>

We check if "SERVICE" item has the same pattern as "FULFILMENTCHARGE" and "WARRANTY".

In [ ]:
# # Number of other items in baskets having SERVICE
# with pl.Config(tbl_rows=-1):
#     display(
#         train
#         .filter(
#             (pl.col('item_clean') == 'SERVICE').any().over('ID')
#         )
#         .filter(
#             pl.col('item_clean') != "SERVICE"
#         )
#         ['item_clean']
#         .value_counts()
#         .sort('count', descending=True)
#     )

<div style="background:AliceBlue">

- "SERVICE" is mostly associated with large items that need installation (TVs, furniture, bedding). Very different profile from "FULFILMENTCHARGE" and "WARRANTY" which were mostly on computers.
- "WARRANTY" appears here in 5th position (286 baskets). This is not contradictory with the previous analysis (where "WARRANTY" was dominated by "COMPUTERS"): 286 baskets is only ~3% of all "WARRANTY" items, so it was invisible in the "COMPUTERS" top.

</div>

Let's check the fraud rate on baskets with "SERVICE" vs without, to see if it's an anti-fraud signal.

In [ ]:
(
    train
    .group_by('ID')
    .agg([
        pl.col('target').first(),
        pl.col('item_clean').is_in(['SERVICE']).any().alias('has_service'),
    ])
    .group_by('has_service')
    .agg([
        pl.len().alias('n_baskets'),
        pl.col('target').mean().alias('fraud_rate') * 100.0,
    ])
)

<div style="background:AliceBlue">

Baskets with "SERVICE" have a fraud rate of 0.10% vs 1.47% without, so almost no fraud when "SERVICE" is present $\implies$ strong anti-fraud indicator.

</div>

#### Study

In [ ]:
prices_combinations = (
    train
        
    # Filter zero prices as we already studied it
    .filter(
        pl.col('cash_price') >= 1.0
    )

    # Add intervals for cash prices
    .with_columns(
        cash_price_interval=numeric_to_intervals(
            pl.col('cash_price'),
            bounds=[0, 10, 20, 50, 100, 500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000],
            inclusive='left',
            fmt='.2fh'
        )
    )

    .group_by('ID', 'target')
    .agg(
        pl.col('n_items').first(),
        pl.col('n_prods').sum(),
        pl.col('model_clean').unique(),
        pl.col('cash_price_interval').unique()
    )
    .with_columns(
        pl.col('cash_price_interval').list.sort()
    )
    .group_by('cash_price_interval', 'model_clean', pl.col('target').cast(pl.Utf8).replace_strict({'1': 'fraud_n_baskets', '0': 'no_fraud_n_baskets'}), pl.col('n_items'), pl.col('n_prods'))
    .agg(
        pl.len().alias('n_baskets')
    )
     .pivot(
            on='target',
            index=['cash_price_interval', 'model_clean', 'n_items', 'n_prods'],
            values='n_baskets'
        )
    .fill_null(0)
    .with_columns(
        (pl.col('fraud_n_baskets') + pl.col('no_fraud_n_baskets')).alias('n_baskets')
    )
    .with_columns(
        pl.col('n_baskets').sum().alias('total_n_baskets')
    )
    .with_columns(
        fraud_rate = pl.col('fraud_n_baskets') / pl.col('n_baskets') * 100.0,
        basket_share = pl.col('n_baskets') / pl.col('total_n_baskets') * 100.0
    )
    .with_columns(
        total_fraud_n_baskets = pl.col('fraud_n_baskets').sum(),
    )
    .with_columns(
        fraud_basket_share = pl.col('fraud_n_baskets') / pl.col('total_fraud_n_baskets') * 100.0
    )
)


In [ ]:
display(prices_combinations.sort('n_baskets', descending=True).head(10))
display(prices_combinations.sort('fraud_n_baskets', descending=True).head(10))

In [ ]:
df = (
    train
    .filter(
        pl.col('cash_price') >= 1.0
    )
    .with_columns(
        pl.when(
            pl.col('model_clean').count().over('model_clean') < 100
        )
        .then(
            pl.lit('OTHER')
        )
        .otherwise(
            pl.col('model_clean')
        )
        .alias('model_clean')
    )
    .with_columns(
        cash_price_interval=numeric_to_intervals(
            pl.col('cash_price'),
            bounds=[0, 10, 20, 50, 100, 500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000],
            inclusive='left',
            fmt='.2fh'
        )
    )
    .group_by('cash_price_interval', pl.col('target').cast(pl.Utf8), pl.col('model_clean'))
    .agg(
        pl.len().alias('n_items')
    )
    .sort('cash_price_interval')
)

px.bar(
    df,
    x='cash_price_interval',
    y='n_items',
    text='n_items',
    facet_col='target',
    color='model_clean',
    title='Distribution of item prices vs target',
).update_yaxes(matches=None, visible=False)

<div style="background:AliceBlue">

- The price range with the highest fraud is the €1k to €2k range, where Apple computers are concentrated.
- For example, 18% of fraudulent baskets involve macbook pros between €1k and €2k (with or without fulfillment charges).
- In the €500 to €1k range, macbook air fraud is also found (5% of fraudulent baskets, 8% fraud rate).
- Many low prices (€0 to €10) are exclusively linked to retailer (and so fulfillment charges as seen before). These prices will be diluted when considering the price per basket.

</div>

#### Item unit prices

*`cash_price` is the total price of each item, not the price of each product within the item. However, we saw each item is generally compound of one product. The unit price study had been carried out but it led to the same conclusions as before, so this section was deleted from the notebook.*

### Basket prices

We can now aggregate prices over baskets to check the distribution, and in particular to see if we are indeed observing price dilution on fulfilment charges items.

In [ ]:
baskets_prices_combinations = (
    train
    .group_by('ID', 'target')
    .agg(
        pl.col('n_prods').sum(),
        pl.col('model_clean').unique(),
        pl.col('cash_price').sum()
    )

    .with_columns(
        cash_price_interval=numeric_to_intervals(
            pl.col('cash_price'),
            bounds=[0, 10, 20, 50, 100, 500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000],
            inclusive='left',
            fmt='.2fh'
        )
    )
    .with_columns(
        pl.col('model_clean').list.sort()
    )
    .group_by('cash_price_interval', 'model_clean', pl.col('target').cast(pl.Utf8).replace_strict({'1': 'fraud_n_baskets', '0': 'no_fraud_n_baskets'}), pl.col('n_prods'))
    .agg(
        pl.len().alias('n_baskets')
    )
     .pivot(
            on='target',
            index=['cash_price_interval', 'model_clean', 'n_prods'],
            values='n_baskets'
        )
    .fill_null(0)
    .with_columns(
        (pl.col('fraud_n_baskets') + pl.col('no_fraud_n_baskets')).alias('n_baskets')
    )
    .with_columns(
        pl.col('n_baskets').sum().alias('total_n_baskets')
    )
    .with_columns(
        fraud_rate = pl.col('fraud_n_baskets') / pl.col('n_baskets') * 100.0,
        basket_share = pl.col('n_baskets') / pl.col('total_n_baskets') * 100.0
    )
    .with_columns(
        total_fraud_n_baskets = pl.col('fraud_n_baskets').sum(),
    )
    .with_columns(
        fraud_basket_share = pl.col('fraud_n_baskets') / pl.col('total_fraud_n_baskets') * 100.0
    )
)


In [ ]:
display(baskets_prices_combinations.sort('n_baskets', descending=True).head(10))
display(baskets_prices_combinations.sort('fraud_n_baskets', descending=True).head(10))

In [ ]:
df = (
    train
    .with_columns(
        pl.when(
            pl.col('make_clean').count().over('make_clean') < 100
        )
        .then(
            pl.lit('OTHER')
        )
        .otherwise(
            pl.col('make_clean')
        )
        .alias('make_clean')
    )
    .group_by('ID', 'target')
    .agg(
        pl.col('cash_price').sum(),
        pl.col('make_clean').unique()
    )
        
    .with_columns(
        cash_price_interval=numeric_to_intervals(
            pl.col('cash_price'),
            bounds=[0, 10, 20, 50, 100, 500, 1_000, 2_000, 5_000, 10_000, 20_000, 50_000],
            inclusive='left',
            fmt='.2fh'
        )
    )
    .group_by('cash_price_interval', pl.col('target').cast(pl.Utf8), pl.col('make_clean'))
    .agg(
        pl.len().alias('n_baskets')
    )
    .sort('cash_price_interval')
    .with_columns(
        pl.col('make_clean').list.sort().list.join(', ')    
    )
        
)

px.bar(
    df,
    x='cash_price_interval',
    y='n_baskets',
    text='n_baskets',
    facet_col='target',
    color='make_clean',
    title='Distribution of baskets prices vs target',
).update_yaxes(matches=None, visible=False)

<div style="background:AliceBlue">

As expected, the price ranges between €0 and €100 are no longer available. Looking at the makes and prices, the conclusion are the same as for prices per item.

</div>

## Feature engineering

### Train set

Based on all the observations done in EDA, we propose this aggregation per basket.

In [ ]:
train_aggregated = (
    train
    .group_by('ID')
    .agg(

        # Basket composition
        pl.col('n_items').first().alias('n_items'),
        pl.col('n_prods').sum().alias('n_prods'),

        # Prices
        pl.col('cash_price').sum().alias('price'),
        (pl.col('cash_price').sum() + 1).log().alias('log_price'),  # price is skewed
        pl.col('cash_price').max().alias('max_price'),
        (pl.col('cash_price').max() + 1).log().alias('log_max_price'),
        pl.col('cash_price').mean().alias('mean_price'),
        (pl.col('cash_price').mean() + 1).log().alias('log_mean_price'),
        pl.col('cash_price').std().fill_null(0.0).alias('std_price'),
        (pl.col('cash_price').std().fill_null(0.0) + 1).log().alias('log_std_price'),
        pl.col('cash_price').max().is_between(1000, 2000).alias('max_price_in_fraud_zone'), 

        # Flags from EDA
        pl.col('item_clean').eq('COMPUTERS').any().alias('has_computer'),                     
        pl.col('item_clean').eq('FULFILMENTCHARGE').any().alias('has_fulfilment'),         
        pl.col('item_clean').eq('SERVICE').any().alias('has_service'),                       
        pl.col('make_clean').eq('APPLE').any().alias('has_apple'),                          
        pl.col('model_clean').str.contains('MACBOOK').any().alias('has_macbook'),
        pl.col('model_clean').str.contains('IPAD').any().alias('has_ipad'),

        pl.col('target').first().alias('target'),
    )
)

### Test set

We redo all the computations done on train set for future inference (with more time, I would have written everything as clean functions).

In [ ]:
x_test = pl.read_csv(r"./data/X_test_8skS2ey.csv", infer_schema_length=None)
raw_x_test_height = x_test.height
print("Original height: ", raw_x_test_height)

# Check no duplicate IDs
print(f"Rows : {x_test.height}")
print(f"Unique IDs : {x_test['ID'].n_unique()}")
print(f"Duplicate IDs : {x_test.height - x_test['ID'].n_unique()}")
assert x_test.height == x_test["ID"].n_unique()

# Unpivot of x_test
x_test = (
    x_test
    .pipe(multi_unpivot)
    .sort('ID', 'item_rank')
)

# Deletion of empty rows
x_test = (
    x_test
    .pipe(remove_empty_items_rows)
)

# Renaming of some columns
x_test = (
    x_test
    .rename(
        dict(
            Nb_of_items="n_items",
            Nbr_of_prod_purchas="n_prods"
        )
    )
)


# Correction of some data types
x_test = (
    x_test
    .with_columns(
        pl.col('ID').cast(pl.Utf8),
        pl.col('n_items').cast(pl.Int64),
        pl.col('n_prods').cast(pl.Int64)
    )
)

# Cleaning of item
x_test = (
    x_test

    # Deletion of characters
    .with_columns(
        pl.col('item')

        # Everyone to uppercase
        .str.to_uppercase()

        # Deletion of any character not being A-Z or number
        .str.replace_all(r'[^A-Z0-9]', '')

        .alias('item_clean')
    )

    # Merge of bathroom items
    .with_columns(
        pl.col('item_clean')
        .replace_strict(
            {
                "BATHROOMACCESSORIES": "BATHROOM",
                "BATHROOMFIXTURES": "BATHROOM"
            },
            default=pl.col('item_clean')
        )
        .alias('item_clean')
    )

    # Replace items starting with digits
    .with_columns(
        pl.when(pl.col('item_clean').str.contains(r'^\d+[A-Z]'))
        .then(pl.lit('OTHER'))
        .otherwise(pl.col('item_clean'))
        .alias('item_clean')
    )

    # Set some outliers to "OTHER"
    .with_columns(
        pl.when(
            pl.col('item_clean').is_in(
                [
                    'APPLEPRODUCTDESCRIPTION',
                    'APPLES',
                    'HPELITEBOOK850V6',
                    'LOGITECHPEBBLEM350BLUETOOTHWIRELESSMOUSE',
                    'MICROSOFTOFFICEHOMEANDSTUDENT2019',
                    'PRODUCT',
                    'TARGUSGEOLITEESSENTIALCASE',
                    'TOSHIBAPORTABLEHARDDRIVE',
                    'UNKNOWN'
                ]
            )
        )
        .then(pl.lit('OTHER'))
        .otherwise(pl.col('item_clean'))
        .alias('item_clean')
    )
)

# Add of make clean
x_test = (
    x_test
    .join(
        make_filling_base
        .select('goods_code', 'make'),
        on='goods_code',
        how='left'
    )
    .with_columns(
        pl.when(
            pl.col('make').is_not_null()
        )
        .then(
            pl.col('make')
        )
        .otherwise(
            pl.col('make_right')
        )
        .fill_null(pl.lit('UNKNOWN'))
        .alias('make_clean')
    )
    .drop('make_right')

    .with_columns(
        pl.col('make_clean')

        # Everyone to uppercase
        .str.to_uppercase()

        # Deletion of any character not being A-Z or number
        .str.replace_all(r'[^A-Z0-9]', '')

        .alias('make_clean')
    )
)

# Add model_clean
x_test = (
    x_test
    .join(
        model_filling_base
        .select('goods_code', 'model'),
        on='goods_code',
        how='left'
    )
    .with_columns(
        pl.when(
            pl.col('model').is_not_null()
        )
        .then(
            pl.col('model')
        )
        .otherwise(
            pl.col('model_right')
        )
        .fill_null(pl.lit('UNKNOWN'))
        .alias('model_clean')
    )
    .drop('model_right')
)

assert x_test['ID'].n_unique() == raw_x_test_height

In [ ]:
x_test_aggregated = (
    x_test
    .group_by('ID')
    .agg(

        # Basket composition
        pl.col('n_items').first().alias('n_items'),
        pl.col('n_prods').sum().alias('n_prods'),
        
        # Prices
        pl.col('cash_price').sum().alias('price'),
        (pl.col('cash_price').sum() + 1).log().alias('log_price'),
        pl.col('cash_price').max().alias('max_price'),
        (pl.col('cash_price').max() + 1).log().alias('log_max_price'),
        pl.col('cash_price').mean().alias('mean_price'),
        (pl.col('cash_price').mean() + 1).log().alias('log_mean_price'),
        pl.col('cash_price').std().fill_null(0.0).alias('std_price'),
        (pl.col('cash_price').std().fill_null(0.0) + 1).log().alias('log_std_price'),
        pl.col('cash_price').max().is_between(1000, 2000).alias('max_price_in_fraud_zone'),
        
        # Flags from EDA
        pl.col('item_clean').eq('COMPUTERS').any().alias('has_computer'),                     
        pl.col('item_clean').eq('FULFILMENTCHARGE').any().alias('has_fulfilment'),         
        pl.col('item_clean').eq('SERVICE').any().alias('has_service'),                       
        pl.col('make_clean').eq('APPLE').any().alias('has_apple'),                          
        pl.col('model_clean').str.contains('MACBOOK').any().alias('has_macbook'),
        pl.col('model_clean').str.contains('IPAD').any().alias('has_ipad'),
    )
)

## Training

The challenge website only allows two submissions per day. Thus, a validation set is defined on the data to enable local testing. However, the data volume (particularly the number of fraudulent baskets) would introduce significant variance in the basket selection, so a cross-validation strategy is implemented.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

# Extract features and target
feature_cols = [c for c in train_aggregated.columns if c not in ['ID', 'target']]
x = train_aggregated.select(feature_cols).to_pandas()
y = train_aggregated['target'].to_pandas()

# Cast booleans to ints
bool_cols = x.select_dtypes('bool').columns
x[bool_cols] = x[bool_cols].astype(int)

# Cross validation
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)

def evaluate(model, x_data, y_data):
    scores = cross_val_score(model, x_data, y_data, cv=kfold, scoring='average_precision')
    return scores

Now we can try to train several models using cross-validation. With more time, I would have improved the feature engineering to test models such as MLPs using PyTorch.

### Logistic Regression

Error when scaling `max_price_in_fraud_zone` which is a boolean. Seen after the last submission on the website challenge.

In [ ]:
%%time
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

features_to_scale = [
    'n_items',
    'n_prods',
    'log_price',
    'log_max_price',
    'log_mean_price',
    'log_std_price',
    'max_price_in_fraud_zone',  # Error here, not changed because I think regression would give bad results in any case
]

other_features = [
    'has_computer',
    'has_fulfilment',
    'has_service',
    'has_apple',
    'has_macbook',
    'has_ipad'
]

features = features_to_scale + other_features

preprocessor = ColumnTransformer(
    transformers=[('scaler', StandardScaler(), features_to_scale)],
    remainder='passthrough'
)

regression = LogisticRegression(solver='lbfgs', fit_intercept=True, class_weight='balanced', max_iter=1000, random_state=0)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', regression),
])

regression_scores = evaluate(pipeline, x[features], y)
print(regression_scores)
print(regression_scores.mean())

### RandomForest

Several hyperparameters have been tested for n_estimators.

In [ ]:
%%time
from sklearn.ensemble import RandomForestClassifier
features = [
    'n_items',
    'n_prods',
    'price',
    'max_price',
    'mean_price',
    'std_price',
    'max_price_in_fraud_zone',
    'has_computer',
    'has_fulfilment',
    'has_service',
    'has_apple',
    'has_macbook',
    'has_ipad'
]

for max_depth in [5, 10, 15, None]:
    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=max_depth,
        class_weight='balanced',
        n_jobs=-1,
        random_state=0
    )
    scores = evaluate(rf, x[features], y)
    print(f"max_depth={max_depth}, scores={scores}, mean={scores.mean():.2f}")

### XGBoost

Same exploration here. Hyperparameters were first tested one by one, then a for loop was used to complete them.

A grid search would have been used if more time had been available.

In [ ]:
%%time
from xgboost import XGBClassifier
features = [
    'n_items',
    'n_prods',
    'price',
    'max_price',
    'mean_price',
    'std_price',
    'max_price_in_fraud_zone',
    'has_computer',
    'has_fulfilment',
    'has_service',
    'has_apple',
    'has_macbook',
    'has_ipad'
]
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

for max_depth in [3, 5, 10]:
    for learning_rate in [0.05, 0.1]:
        xgb = XGBClassifier(
            n_estimators=500,
            max_depth=max_depth,
            learning_rate=learning_rate,
            min_child_weight=5,
            scale_pos_weight=scale_pos_weight,
            eval_metric='aucpr',
            tree_method='hist',
            n_jobs=-1,
            random_state=0,
        )
        scores = evaluate(xgb, x[features], y)
        print(f"max_depth={max_depth}, learning_rate={learning_rate}, scores={scores}, mean={scores.mean():.2f}")

### Conclusion

Best model is XGBoost with:
- n_estimators = 500
- max_depth = 10
- learning_rate = 0.05
- min_child_weight = 5

*To avoid having to rewrite the model, the best model should be saved directly in the training loops.*

## Inference

We retrain the best architecture on all the train set and evaluate it on the test data for submission.

In [ ]:
%%time
from sklearn.metrics import average_precision_score

# Prepare x_test
x_test_df = x_test_aggregated.select(['ID'] + features).to_pandas()
bool_cols_test = x_test_df.select_dtypes('bool').columns
x_test_df[bool_cols_test] = x_test_df[bool_cols_test].astype(int)

# Best model trained on all train data
scale_pos_weight = (y == 0).sum() / (y == 1).sum()
final_model = XGBClassifier(
    n_estimators=500,
    max_depth=10,
    learning_rate=0.05,
    min_child_weight=5,
    scale_pos_weight=scale_pos_weight,
    eval_metric='aucpr',
    tree_method='hist',
    n_jobs=-1,
    random_state=0,
)
final_model.fit(x[features], y)

# Metric on the training data
train_probas = final_model.predict_proba(x[features])[:, 1]
train_pr_auc = average_precision_score(y, train_probas)
print(f"Train PR-AUC: {train_pr_auc:.2f}")

# Prediction on the test set
probas = final_model.predict_proba(x_test_df[features])[:, 1]

# Build of the final dataframe for submission
submission = x_test_df[['ID']].copy()
submission['ID'] = submission['ID'].astype(int)
submission['fraud_flag'] = probas
y_test_random = pl.read_csv(r"./data/Y_test_random_2.csv").to_pandas()
submission = y_test_random[['ID']].merge(submission, on='ID', how='left')

# Save as CSV file
submission.to_csv('submission.csv', index=True, index_label='index')

## Semi-supervised training

In this section, we try a semi-supervised training approach consisting in :
- Training the previous best model on training data
- Predicting on test data
- Move best predictions on test data to training data
- Training the model on those new training data
- Etc, until no prediction is good enough to be moved to training data

In [ ]:
def get_model(y_df):
    scale_pos_weight = (y_df == 0).sum() / (y_df == 1).sum()
    best_model = XGBClassifier(
        n_estimators=500,
        max_depth=10,
        learning_rate=0.05,
        min_child_weight=5,
        scale_pos_weight=scale_pos_weight,
        eval_metric='aucpr',
        tree_method='hist',
        n_jobs=-1,
        random_state=0,
    )
    return best_model

In [ ]:
high_threshold = 0.9
low_threshold = 0.1
x_semi_train = x[features].copy().reset_index(drop=True)
y_semi_train = y.copy().reset_index(drop=True)
x_semi_test = x_test_df.copy().reset_index(drop=True)

while True:
    model = get_model(y_semi_train)
    model.fit(x_semi_train[features], y_semi_train)
    y_semi_test = model.predict_proba(x_semi_test[features])[:, 1]
    
    confident_high = y_semi_test >= high_threshold
    confident_low = y_semi_test <= low_threshold
    confident_mask = confident_high | confident_low
    
    if not confident_mask.any():
        break
    
    pseudo_y = pd.Series(np.where(confident_high[confident_mask], 1, 0))
    pseudo_x = x_semi_test.loc[confident_mask].reset_index(drop=True)
    
    x_semi_train = pd.concat([x_semi_train, pseudo_x], ignore_index=True)
    y_semi_train = pd.concat([y_semi_train, pseudo_y], ignore_index=True)
    
    x_semi_test = x_semi_test.loc[~confident_mask].reset_index(drop=True)

result_df = x_test_df.copy().reset_index(drop=True)
result_df['fraud_flag'] = model.predict_proba(result_df[features])[:, 1]

In [ ]:
# Build of the final dataframe for submission
result_df['ID'] = result_df['ID'].astype(int)
submission = y_test_random[['ID']].merge(result_df[['ID', 'fraud_flag']], on='ID', how='left')

# Save as CSV file
submission.to_csv('submission_semi.csv', index=True, index_label='index')